<a href="https://colab.research.google.com/github/mochwendy/sentimen-app/blob/main/Portofolio_AI_Engineer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Memanggil model ringkasan teks dari Hugging Face
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2. Masukkan teks panjang yang ingin dirangkum (Contoh teks bahasa Inggris)
teks_panjang = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the intelligence of humans and other animals.
AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars,
and automated decision-making. As machines become increasingly capable, tasks considered to require "intelligence" are often
removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is
frequently excluded from things considered to be AI, having become a routine technology.
"""

# 3. Jalankan proses AI
inputs = tokenizer(teks_panjang, max_length=1024, truncation=True, return_tensors="pt")
summary_ids = model.generate(
    inputs["input_ids"], num_beams=4, max_length=50, min_length=10, early_stopping=True
)
hasil_ringkasan = tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# 4. Tampilkan hasil ringkasan
print("Hasil Ringkasan AI:")
print(hasil_ringkasan)

In [ ]:
from transformers import pipeline

# 1. Ganti tugas menjadi "text-generation" agar sesuai dengan library versi terbaru
perangkum = pipeline("text-generation", model="facebook/bart-large-cnn")

# 2. Masukkan teks panjang yang ingin dirangkum
teks_panjang = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the intelligence of humans and other animals.
AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars,
and automated decision-making. As machines become increasingly capable, tasks considered to require "intelligence" are often
removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is
frequently excluded from things considered to be AI, having become a routine technology.
"""

# 3. Jalankan proses AI (Ganti max_length menjadi max_new_tokens agar kompatibel)
hasil = perangkum(teks_panjang, max_new_tokens=50, do_sample=False)

# 4. Tampilkan hasil ringkasan
print("Hasil Ringkasan AI:")
print(hasil[0]['generated_text'])


In [ ]:
# 1. Panggil komponen spesifik untuk peringkasan teks
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/bart-large-cnn"

# 2. Muat model dan pengolah teks (tokenizer) sesuai arsitektur aslinya
tokenizer = AutoTokenizer.from_pretrained(model_name, clean_up_tokenization_spaces=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Teks panjang yang ingin dirangkum
teks_panjang = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the intelligence of humans and other animals.
AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars,
and automated decision-making. As machines become increasingly capable, tasks considered to require "intelligence" are often
removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is
frequently excluded from things considered to be AI, having become a routine technology.
"""

# 4. Ubah teks menjadi angka (token) yang dipahami oleh AI
inputs = tokenizer(teks_panjang, return_tensors="pt", max_length=1024, truncation=True)

# 5. Jalankan model AI untuk merangkum teks
summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=10, length_penalty=2.0, num_beams=4, early_stopping=True)

# 6. Ubah kembali angka dari AI menjadi teks manusia yang bersih
hasil_bersih = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Hasil Ringkasan AI yang Benar:")
print(hasil_bersih)


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# 1. Tentukan nama model analisis sentimen dari Hugging Face
nama_model = "distilbert-base-uncased-finetuned-sst-2-english"

# 2. Muat pengolah teks (tokenizer) dan model AI
pengolah_teks = AutoTokenizer.from_pretrained(nama_model)
model_emosi = AutoModelForSequenceClassification.from_pretrained(nama_model)

# 3. MASUKKAN KALIMAT YANG INGIN DIUJI DI SINI (Gunakan Bahasa Inggris)
kalimat_uji = "I am so frustrated because the system keeps crashing and I lost all my unsaved progress."

# 4. Ubah kalimat menjadi format angka yang dipahami AI
input_data = pengolah_teks(kalimat_uji, return_tensors="pt")

# 5. Jalankan AI untuk memprediksi emosi kalimat
with torch.no_grad():
    output = model_emosi(**input_data)

# 6. Hitung hasil skor prediksi
prediksi = torch.nn.functional.softmax(output.logits, dim=-1)
skor_tertinggi = torch.argmax(prediksi).item()
persentase = prediksi[0][skor_tertinggi].item() * 100

# 7. Tampilkan hasil tebakan AI ke layar
label_emosi = "POSITIF (Senang/Pujian)" if skor_tertinggi == 1 else "NEGATIF (Sedih/Kecewa/Marah)"

print("Kalimat Anda:")
print(f'"{kalimat_uji}"\n')
print(f"Hasil Analisis AI: Kalimat ini bermakna {label_emosi}")
print(f"Tingkat Keyakinan AI: {persentase:.2f}%")


In [ ]:
!pip install streamlit transformers torch -q
!npm install -g localtunnel -q


In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

st.set_page_config(page_title="AI Sentiment Analyzer", page_icon="🤖")
st.title("🤖 AI Pendeteksi Emosi Teks")
st.write("Masukkan kalimat dalam Bahasa Inggris untuk menganalisis apakah maknanya Positif atau Negatif.")

@st.cache_resource
def load_model():
    nama_model = "distilbert-base-uncased-finetuned-sst-2-english"
    pengolah_teks = AutoTokenizer.from_pretrained(nama_model)
    model_emosi = AutoModelForSequenceClassification.from_pretrained(nama_model)
    return pengolah_teks, model_emosi

pengolah_teks, model_emosi = load_model()

kalimat_uji = st.text_area("Tulis kalimat Anda di sini:", "I am very happy to learn AI engineering today!")

if st.button("Analisis Sentimen"):
    with st.spinner("AI sedang berpikir..."):
        input_data = pengolah_teks(kalimat_uji, return_tensors="pt")
        with torch.no_grad():
            output = model_emosi(**input_data)

        prediksi = torch.nn.functional.softmax(output.logits, dim=-1)
        skor_tertinggi = torch.argmax(prediksi).item()
        persentase = prediksi[skor_tertinggi].item() * 100

        st.write("---")
        if skor_tertinggi == 1:
            st.success(f"**Hasil:** POSITIF (Senang/Pujian) 🎉")
        else:
            st.error(f"**Hasil:** NEGATIF (Sedih/Kecewa/Marah) ⚠️")

        st.info(f"**Tingkat Keyakinan AI:** {persentase:.2f}%")



In [ ]:
!wget -qO- ://icanhazip.com


In [ ]:
!wget -qO- ://icanhazip.com

In [ ]:
!curl ipecho.net/plain; echo
!streamlit run app.py & npx localtunnel --port 8501



In [ ]:
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 py@pinggy.io


In [ ]:
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 py@pinggy.io



In [ ]:
!ssh -p 443 -o StrictHostKeyChecking=no -R 80:localhost:8501 free@a.pinggy.io


In [ ]:
import urllib.request
import json

# Mengambil info tautan aktif dari lokal server Pinggy
try:
    with urllib.request.urlopen("http://localhost:4321/urls") as url:
        data = json.loads(url.read().decode())
        print("🎉 BERHASIL! Klik tautan di bawah ini untuk membuka Web AI Anda:")
        print("----------------------------------------------------------------")
        print(data['urls'][0])
        print("----------------------------------------------------------------")
except Exception:
    print("Gagal mengambil link otomatis. Silakan cek kembali kotak kode SSH di atas,")
    print("biasanya ada link berakhiran '.pinggy.link' yang bisa langsung diklik.")


In [ ]:
from google.colab import output
import time

# 1. Jalankan server Streamlit di latar belakang pada port 8501
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

# 2. Tunggu 3 detik agar server Streamlit siap
time.sleep(3)

# 3. Tampilkan antarmuka web Streamlit langsung di dalam halaman Colab
output.serve_kernel_port_as_iframe(8501, height='600')


In [ ]:
# 1. Matikan paksa semua proses streamlit yang menggantung sebelumnya
!pkill streamlit
import time

# 2. Jalankan ulang Streamlit dengan konfigurasi headless (tanpa browser lokal)
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 3. Berikan waktu lebih lama (5 detik) agar server benar-benar siap
print("Sedang menyalakan server Streamlit, mohon tunggu...")
time.sleep(5)

# 4. Tampilkan kembali jendela aplikasi web
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height='600')


In [ ]:
!streamlit run app.py --server.port 8501


In [ ]:
from google.colab import output
import time

# 1. Jalankan server Streamlit di port baru (8502)
get_ipython().system_raw('streamlit run app.py --server.port 8502 --server.headless true &')

# 2. Tunggu 5 detik agar server siap
print("Sedang mengalihkan ke Port 8502, mohon tunggu...")
time.sleep(5)

# 3. Tampilkan aplikasi web dari port 8502
output.serve_kernel_port_as_iframe(8502, height='600')


In [ ]:
from pyngrok import ngrok
import time

# 1. Masukkan token autentikasi Anda ke dalam sistem Ngrok
ngrok.set_auth_token("3FZJvkVJemqrHr8Wuko2P26zVgz")

# 2. Bersihkan sisa proses lama & jalankan ulang Streamlit di latar belakang
!pkill streamlit
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 3. Beri waktu 3 detik agar server Streamlit siap sepenuhnya
print("Sedang menyiapkan jalur server, mohon tunggu...")
time.sleep(3)

# 4. Buka terowongan publik Ngrok untuk meneruskan port 8501
try:
    # Putus koneksi lama jika ada yang menggantung
    ngrok.disconnect(ngrok.connect(8501).public_url)
except:
    pass

tautan_web = ngrok.connect(8501)

# 5. Cetak tautan resmi untuk Anda klik
print("\n🎉 BERHASIL! Klik tautan resmi di bawah ini untuk membuka Web AI Anda:")
print("---------------------------------------------------------------------")
print(tautan_web.public_url)
print("---------------------------------------------------------------------")
print("Catatan: Jika muncul halaman peringatan dari Ngrok, cukup klik tombol 'Visit Site'.")


In [ ]:
# 1. Install library pyngrok langsung di awal
!pip install pyngrok -q

from pyngrok import ngrok
import time

# 2. Masukkan token autentikasi Anda
ngrok.set_auth_token("3FZJvkVJemqrHr8Wuko2P26zVgz")

# 3. Bersihkan sisa proses lama & jalankan ulang Streamlit di latar belakang
!pkill streamlit
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 4. Beri waktu 3 detik agar server Streamlit siap sepenuhnya
print("Sedang menyiapkan jalur server, mohon tunggu...")
time.sleep(3)

# 5. Buka terowongan publik Ngrok untuk meneruskan port 8501
try:
    ngrok.kill() # Membersihkan sisa koneksi terowongan ngrok terdahulu
except:
    pass

tautan_web = ngrok.connect(8501)

# 6. Cetak tautan resmi untuk Anda klik
print("\n🎉 BERHASIL! Klik tautan resmi di bawah ini untuk membuka Web AI Anda:")
print("---------------------------------------------------------------------")
print(tautan_web.public_url)
print("---------------------------------------------------------------------")
print("Catatan: Jika muncul halaman peringatan dari Ngrok, cukup klik tombol 'Visit Site'.")


In [ ]:
# 1. Pastikan library pyngrok terinstal sempurna
!pip install pyngrok -q

from pyngrok import ngrok
import time

# 2. Masukkan token autentikasi resmi Anda yang baru
ngrok.set_auth_token("3FZKaN9J5groF8WYi9Lr7RUBHPK_6b4YHHR6wwvAaPLTjftj5")

# 3. Matikan semua sisa proses lama agar port tidak tersumbat
!pkill streamlit
try:
    ngrok.kill()
except:
    pass

# 4. Jalankan server Streamlit di latar belakang (Port 8501)
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true &')

# 5. Beri jeda 3 detik untuk aktivasi sistem
print("Sedang menyambungkan terowongan Ngrok, mohon tunggu...")
time.sleep(3)

# 6. Buka jalur publik menggunakan protokol HTTP resmi
tautan_web = ngrok.connect(8501, proto="http")

# 7. Cetak link web akhir untuk Anda
print("\n🎉 SUKSES BESAR! Web AI Anda telah online di seluruh dunia:")
print("---------------------------------------------------------------------")
print(tautan_web.public_url)
print("---------------------------------------------------------------------")
print("Catatan: Jika muncul halaman 'ngrok: Welcome', cukup klik tombol 'Visit Site'.")


In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# 1. Judul Aplikasi Web
st.set_page_config(page_title="AI Sentiment Analyzer", page_icon="🤖")
st.title("🤖 AI Pendeteksi Emosi Teks")
st.write("Masukkan kalimat dalam Bahasa Inggris untuk menganalisis apakah maknanya Positif atau Negatif.")

# 2. Load Model AI (Gunakan cache)
@st.cache_resource
def load_model():
    nama_model = "distilbert-base-uncased-finetuned-sst-2-english"
    pengolah_teks = AutoTokenizer.from_pretrained(nama_model)
    model_emosi = AutoModelForSequenceClassification.from_pretrained(nama_model)
    return pengolah_teks, model_emosi

pengolah_teks, model_emosi = load_model()

# 3. Membuat Input Teks di Web
kalimat_uji = st.text_area("Tulis kalimat Anda di sini:", "I am very happy to learn AI engineering today!")

# 4. Membuat Tombol Jalankan
if st.button("Analisis Sentimen"):
    with st.spinner("AI sedang berpikir..."):
        # Jalankan Proses AI
        input_data = pengolah_teks(kalimat_uji, return_tensors="pt")
        with torch.no_grad():
            output = model_emosi(**input_data)

        # Hitung Hasil Skor Prediksi (Ditambahkan [0] untuk mengambil dimensi pertama)
        prediksi = torch.nn.functional.softmax(output.logits, dim=-1)[0]
        skor_tertinggi = torch.argmax(prediksi).item()
        persentase = prediksi[skor_tertinggi].item() * 100

        # Tampilkan Hasil di Web
        st.write("---")
        if skor_tertinggi == 1:
            st.success(f"**Hasil:** POSITIF (Senang/Pujian) 🎉")
        else:
            st.error(f"**Hasil:** NEGATIF (Sedih/Kecewa/Marah) ⚠️")

        st.info(f"**Tingkat Keyakinan AI:** {persentase:.2f}%")


In [ ]:
%%writefile aturan_kantor.txt
Aturan Resmi PT AI CINTA KARYA KITA:
1. Jam kerja kantor dimulai tepat pukul 08.00 WIB dan berakhir pada pukul 17.00 WIB.
2. Setiap karyawan berhak mendapatkan jatah cuti tahunan sebanyak 12 hari kerja.
3. Fasilitas makan siang gratis disediakan oleh perusahaan khusus pada hari Jumat.
4. Pakaian pada hari Senin sampai Kamis adalah kemeja rapi, sedangkan hari Jumat diperbolehkan memakai baju batik atau kaos berkerah.


In [ ]:
import gradio as gr
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Memuat dokumen internal yang sudah dibuat
loader = TextLoader("aturan_kantor.txt", encoding="utf-8")
dokumen = loader.load()

# 2. Memotong teks panjang menjadi bagian-bagian kecil agar mudah dipahami AI
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
potongan_teks = text_splitter.split_documents(dokumen)

# 3. Menggunakan model Hugging Face khusus bahasa Indonesia untuk menyamakan arti kata (Embeddings)
print("Sedang memuat model kecerdasan bahasa, mohon tunggu...")
model_embedding = HuggingFaceEmbeddings(model_name="indobenchmark/indobert-base-p2")

# 4. Menyimpan pengetahuan dokumen ke dalam Database Vektor (ChromaDB) di memori komputer
database_vektor = Chroma.from_documents(potongan_teks, model_embedding)

# 5. Fungsi Utama RAG untuk mencari jawaban paling cocok di dalam dokumen
def tanya_ai_rag(pertanyaan_user):
    # Mencari potongan dokumen yang paling mirip dengan pertanyaan
    hasil_pencarian = database_vektor.similarity_search(pertanyaan_user, k=1)

    if hasil_pencarian:
        return f"📖 Hasil Temuan AI di Dokumen:\n\n{hasil_pencarian[0].page_content}"
    else:
        return "Maaf, informasi tersebut tidak ditemukan dalam dokumen internal."

# 6. Membuat Tampilan Web Interaktif dengan Gradio
antarmuka_rag = gr.Interface(
    fn=tanya_ai_rag,
    inputs=gr.Textbox(lines=2, placeholder="Contoh: Jam berapa kantor dimulai? / Berapa hari jatah cuti?", label="Pertanyaan Anda"),
    outputs=gr.Textbox(label="Jawaban AI berdasarkan Dokumen"),
    title="🏢 AI Asisten Dokumen Perusahaan (RAG)",
    description="AI ini akan menjawab pertanyaan Anda murni berdasarkan dokumen aturan internal perusahaan."
)

# 7. Jalankan web dan buat tautan publik otomatis
antarmuka_rag.launch(share=True, debug=True)


In [ ]:
# 1. Install ulang seluruh library pendukung secara paksa dan eksplisit
!pip install langchain langchain-community langchain-text-splitters chromadb pypdf gradio sentence-transformers -q

import gradio as gr
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

print("✅ Semua library berhasil dimuat!")

# 2. Pastikan file aturan_kantor.txt ada
if not os.path.exists("aturan_kantor.txt"):
    with open("aturan_kantor.txt", "w", encoding="utf-8") as f:
        f.write("""Aturan Resmi PT AI Maju Bersama:
1. Jam kerja kantor dimulai tepat pukul 08.00 WIB dan berakhir pada pukul 17.00 WIB.
2. Setiap karyawan berhak mendapatkan jatah cuti tahunan sebanyak 12 hari kerja.
3. Fasilitas makan siang gratis disediakan oleh perusahaan khusus pada hari Jumat.
4. Pakaian pada hari Senin sampai Kamis adalah kemeja rapi, sedangkan hari Jumat diperbolehkan memakai baju batik atau kaos berkerah.""")

# 3. Memuat dan memotong dokumen internal
loader = TextLoader("aturan_kantor.txt", encoding="utf-8")
dokumen = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
potongan_teks = text_splitter.split_documents(dokumen)

# 4. Memuat model kecerdasan bahasa Indonesia (Embeddings)
print("⏳ Sedang mengunduh model bahasa Indonesia (indobert), mohon tunggu...")
model_embedding = HuggingFaceEmbeddings(model_name="indobenchmark/indobert-base-p2")

# 5. Menyimpan ke Database Vektor (ChromaDB)
database_vektor = Chroma.from_documents(potongan_teks, model_embedding)

# 6. Fungsi Utama pencarian dokumen RAG
def tanya_ai_rag(pertanyaan_user):
    hasil_pencarian = database_vektor.similarity_search(pertanyaan_user, k=1)
    if hasil_pencarian:
        return f"📖 Hasil Temuan AI di Dokumen:\n\n{hasil_pencarian[0].page_content}"
    else:
        return "Maaf, informasi tersebut tidak ditemukan dalam dokumen internal."

# 7. Membuat dan Menjalankan Tampilan Web dengan Gradio
antarmuka_rag = gr.Interface(
    fn=tanya_ai_rag,
    inputs=gr.Textbox(lines=2, placeholder="Contoh: Jam berapa kantor dimulai? / Hari apa ada makan siang gratis?", label="Pertanyaan Anda"),
    outputs=gr.Textbox(label="Jawaban AI berdasarkan Dokumen"),
    title="🏢 AI Asisten Dokumen Perusahaan (RAG Bahasa Indonesia)",
    description="AI ini akan menjawab pertanyaan Anda murni berdasarkan dokumen aturan internal perusahaan."
)

# Jalankan web dan buat tautan publik otomatis
antarmuka_rag.launch(share=True, debug=True)
